In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, hamming_loss, f1_score

# استيراد الخوارزميات الخمس بالإعدادات الكاملة
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

warnings.filterwarnings('ignore')

# -------------------------------------------------------------
# 1. بناء فئة MLSMOTE المباشرة للـ Multi-Label
# -------------------------------------------------------------
class MLSMOTE:
    def __init__(self, k_neighbors=5, random_state=42):
        self.k_neighbors = k_neighbors
        self.random_state = random_state

    def fit_resample(self, X, y):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape

        k = min(self.k_neighbors, n_samples - 1)
        if k < 1:
            return X, y

        knn = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1).fit(X)
        indices = knn.kneighbors(X, return_distance=False)

        synthetic_X, synthetic_y = [], []

        for i in range(n_samples):
            neighbor_idx = np.random.choice(indices[i][1:])
            step = np.random.rand()

            x_new = X[i] + step * (X[neighbor_idx] - X[i])
            y_new = np.bitwise_or(y[i].astype(int), y[neighbor_idx].astype(int))

            synthetic_X.append(x_new)
            synthetic_y.append(y_new)

        X_resampled = np.vstack([X, np.array(synthetic_X)])
        y_resampled = np.vstack([y, np.array(synthetic_y)])

        # التأكد القاطع من أن الوسوم الاصطناعية ثنائية 0 أو 1 فقط
        y_resampled = (y_resampled > 0).astype(np.int32)

        return X_resampled, y_resampled

# -------------------------------------------------------------
# 2. النماذج بالأداء الكامل والدقة الأصلية (100 شجرة وطبقات كاملة)
# -------------------------------------------------------------
models = {
    "1. Multi-Layer Perceptron (MLP)": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=300,
        random_state=42
    ),
    "2. Decision Tree (Direct)": DecisionTreeClassifier(random_state=42),
    "3. Random Forest (Direct)": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "4. Extra Trees (Direct)": ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "5. K-Nearest Neighbors (Direct Adaptation)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

# -------------------------------------------------------------
# 3. قاموس ملفات CSV وعدد الوسوم (Labels) لكل داتا
# -------------------------------------------------------------
datasets_config = {
    'yeast': {'path': 'yeast_dataset.csv', 'num_labels': 14},
    'emotions': {'path': 'emotions_dataset.csv', 'num_labels': 6},
    'scene': {'path': 'scene_dataset.csv', 'num_labels': 6},
    'bookmarks': {'path': 'bookmarks_dataset.csv', 'num_labels': 208},
    'genbase': {'path': 'genbase_dataset.csv', 'num_labels': 27},
    'flags': {'path': 'flags.csv', 'num_labels': 7}
}

# -------------------------------------------------------------
# 4. الحلقة التكرارية لجميع الملفات والنماذج
# -------------------------------------------------------------
for ds_name, config in datasets_config.items():
    print("=" * 65)
    print(f" 📊 جاري تحليل وتطبيق الخوارزميات على داتا: [{ds_name.upper()}]")
    print("=" * 65)

    try:
        df = pd.read_csv(config['path'])
    except FileNotFoundError:
        print(f"⚠️ تعذر العثور على الملف: {config['path']}، سيتم تخطيه.\n")
        continue

    num_labels = config['num_labels']

    X = df.iloc[:, :-num_labels].values.astype(np.float32)
    # تحويل صريح وشرطي للوسوم إلى (0 أو 1) فقط لمنع خطأ Binarization في MLP
    y = (df.iloc[:, -num_labels:].values > 0).astype(np.int32)

    # تقسيم البيانات (80% تدريب - 20% اختبار)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"• عينات التدريب الأصلية: {X_train.shape[0]} | عينات الاختبار: {X_test.shape[0]}")

    # تطبيق ML-SMOTE المباشر
    try:
        mlsmote = MLSMOTE(k_neighbors=5, random_state=42)
        X_train_res, y_train_res = mlsmote.fit_resample(X_train, y_train)
        print(f"• عينات التدريب بعد ML-SMOTE المباشر: {X_train_res.shape[0]} (تم توليد عينات متوازنة)")
    except Exception as e:
        X_train_res, y_train_res = X_train, y_train
        print(f"• تعذر تطبيق ML-SMOTE المباشر: {e}")

    print(f"• عدد الميزات (Features): {X_train.shape[1]} | عدد الوسوم (Labels): {y_train.shape[1]}\n")

    # تطبيق الخوارزميات الخمس
    for model_name, model in models.items():
        model.fit(X_train_res, y_train_res)
        y_pred = model.predict(X_test)

        if hasattr(y_pred, 'toarray'):
            y_pred = y_pred.toarray()

        acc = accuracy_score(y_test, y_pred)
        h_loss = hamming_loss(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro')

        print(f"🔹 {model_name}:")
        print(f"   ├─ Subset Accuracy : {acc * 100:.2f}%")
        print(f"   ├─ Hamming Loss     : {h_loss:.4f}")
        print(f"   └─ Macro F1-Score   : {f1_macro:.4f}")
        print("-" * 50)

    print("\n")

 📊 جاري تحليل وتطبيق الخوارزميات على داتا: [YEAST]
• عينات التدريب الأصلية: 1933 | عينات الاختبار: 484
• عينات التدريب بعد ML-SMOTE المباشر: 3866 (تم توليد عينات متوازنة)
• عدد الميزات (Features): 103 | عدد الوسوم (Labels): 14

🔹 1. Multi-Layer Perceptron (MLP):
   ├─ Subset Accuracy : 9.30%
   ├─ Hamming Loss     : 0.2521
   └─ Macro F1-Score   : 0.4398
--------------------------------------------------
🔹 2. Decision Tree (Direct):
   ├─ Subset Accuracy : 9.50%
   ├─ Hamming Loss     : 0.2980
   └─ Macro F1-Score   : 0.4134
--------------------------------------------------
🔹 3. Random Forest (Direct):
   ├─ Subset Accuracy : 20.45%
   ├─ Hamming Loss     : 0.1871
   └─ Macro F1-Score   : 0.4168
--------------------------------------------------
🔹 4. Extra Trees (Direct):
   ├─ Subset Accuracy : 18.60%
   ├─ Hamming Loss     : 0.1933
   └─ Macro F1-Score   : 0.3944
--------------------------------------------------
🔹 5. K-Nearest Neighbors (Direct Adaptation):
   ├─ Subset Accuracy : 